In [1]:
print('hello world')

hello world


In [ ]:
import pandas as pd
import torch
import random
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import pandas as pd

In [3]:
# Team and venue data
teams = [
    "Mumbai Indians",
    "Chennai Super Kings",
    "Royal Challengers Bangalore",
    "Kolkata Knight Riders",
    "Delhi Capitals",
    "Rajasthan Royals",
    "Punjab Kings",
    "Sunrisers Hyderabad"
]

venues = [
    "Wankhede Stadium",
    "Chepauk Stadium",
    "M Chinnaswamy Stadium",
    "Eden Gardens",
    "Arun Jaitley Stadium",
    "Sawai Mansingh Stadium",
    "IS Bindra Stadium",
    "Rajiv Gandhi Stadium"
]

toss_decisions = ["bat", "field"]

# Generate dataset
data = []
num_matches = 200
for _ in range(num_matches):
    team1, team2 = random.sample(teams, 2)
    toss_winner = random.choice([team1, team2])
    toss_decision = random.choice(toss_decisions)
    venue = random.choice(venues)
    winner = random.choice([team1, team2])
    data.append([team1, team2, toss_winner, toss_decision, venue, winner])

df = pd.DataFrame(data, columns=[
    "team1", "team2", "toss_winner", "toss_decision", "venue", "winner"
])
df.to_csv("matches.csv", index=False)
print("Dataset created successfully ")

# Load and preprocess data
df = pd.read_csv("matches.csv")
X = df.drop("winner", axis=1)
y = df["winner"]

encoders = {}
for col in X.columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

# Define and train model
class IPLModel(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.out = nn.Linear(16, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        return self.out(x)

model = IPLModel(X_train.shape[1], len(set(y)))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Training model...")
for epoch in range(50):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)
print(f"\nModel Accuracy: {accuracy*100:.2f}%")

# Prediction function
def predict_match(team1, team2, toss_winner, toss_decision, venue):
    input_data = pd.DataFrame([[team1, team2, toss_winner, toss_decision, venue]],
                              columns=X.columns)
    for col in input_data.columns:
        input_data[col] = encoders[col].transform(input_data[col])
    input_data = scaler.transform(input_data)
    input_tensor = torch.tensor(input_data, dtype=torch.float32)
    
    with torch.no_grad():
        output = model(input_tensor)
        _, pred = torch.max(output, 1)
    return target_encoder.inverse_transform(pred.numpy())[0]

# Tkinter GUI
class IPLPredictorApp:
    def __init__(self, root):
        self.root = root
        self.root.title("🏏 IPL Match Winner Predictor")
        self.root.geometry("1400x900")
        self.root.configure(bg="#0D1117")
        
        # Title
        title_frame = tk.Frame(root, bg="#0D1117")
        title_frame.pack(pady=20)
        title_label = tk.Label(title_frame, text="🏏 IPL MATCH WINNER PREDICTOR", 
                              font=("Arial", 28, "bold"), fg="#00D4AA", bg="#0D1117")
        title_label.pack()
        subtitle_label = tk.Label(title_frame, text=f"Model Accuracy: {accuracy*100:.1f}%", 
                                 font=("Arial", 14), fg="#58A6FF", bg="#0D1117")
        subtitle_label.pack()
        
        # Main container
        main_frame = tk.Frame(root, bg="#0D1117")
        main_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=10)
        
        # Left panel - Prediction form
        left_frame = tk.Frame(main_frame, bg="#161B22", relief=tk.RAISED, bd=2)
        left_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=False, padx=(0, 15), pady=0)
        left_frame.pack_propagate(False)
        left_frame.configure(width=450)
        
        # Prediction form
        form_frame = tk.LabelFrame(left_frame, text="📋 Match Details", 
                                  font=("Arial", 14, "bold"), fg="white", bg="#161B22")
        form_frame.pack(fill=tk.X, padx=20, pady=20)
        
        # Team 1
        tk.Label(form_frame, text="Team 1:", font=("Arial", 12), fg="white", bg="#161B22").grid(row=0, column=0, sticky=tk.W, padx=10, pady=10)
        self.team1_var = tk.StringVar(value="Mumbai Indians")
        team1_combo = ttk.Combobox(form_frame, textvariable=self.team1_var, values=teams, 
                                  state="readonly", font=("Arial", 11))
        team1_combo.grid(row=0, column=1, sticky=tk.W, padx=10, pady=10)
        
        # Team 2
        tk.Label(form_frame, text="Team 2:", font=("Arial", 12), fg="white", bg="#161B22").grid(row=1, column=0, sticky=tk.W, padx=10, pady=10)
        self.team2_var = tk.StringVar(value="Chennai Super Kings")
        team2_combo = ttk.Combobox(form_frame, textvariable=self.team2_var, values=teams, 
                                  state="readonly", font=("Arial", 11))
        team2_combo.grid(row=1, column=1, sticky=tk.W, padx=10, pady=10)
        
        # Toss winner
        tk.Label(form_frame, text="Toss Winner:", font=("Arial", 12), fg="white", bg="#161B22").grid(row=2, column=0, sticky=tk.W, padx=10, pady=10)
        self.toss_var = tk.StringVar(value="Mumbai Indians")
        toss_combo = ttk.Combobox(form_frame, textvariable=self.toss_var, values=teams, 
                                 state="readonly", font=("Arial", 11))
        toss_combo.grid(row=2, column=1, sticky=tk.W, padx=10, pady=10)
        
        # Toss decision
        tk.Label(form_frame, text="Toss Decision:", font=("Arial", 12), fg="white", bg="#161B22").grid(row=3, column=0, sticky=tk.W, padx=10, pady=10)
        self.toss_decision_var = tk.StringVar(value="bat")
        toss_dec_combo = ttk.Combobox(form_frame, textvariable=self.toss_decision_var, 
                                     values=toss_decisions, state="readonly", font=("Arial", 11))
        toss_dec_combo.grid(row=3, column=1, sticky=tk.W, padx=10, pady=10)
        
        # Venue
        tk.Label(form_frame, text="Venue:", font=("Arial", 12), fg="white", bg="#161B22").grid(row=4, column=0, sticky=tk.W, padx=10, pady=10)
        self.venue_var = tk.StringVar(value="Wankhede Stadium")
        venue_combo = ttk.Combobox(form_frame, textvariable=self.venue_var, values=venues, 
                                  state="readonly", font=("Arial", 11))
        venue_combo.grid(row=4, column=1, sticky=tk.W, padx=10, pady=10)
        
        # Predict button
        predict_btn = tk.Button(form_frame, text="🎯 PREDICT WINNER", command=self.predict_match,
                               bg="#00D4AA", fg="black", font=("Arial", 14, "bold"), 
                               relief=tk.RAISED, bd=3, pady=10, cursor="hand2")
        predict_btn.grid(row=5, column=0, columnspan=2, pady=25)
        
        # Prediction result
        self.result_label = tk.Label(left_frame, text="👆 Enter match details and predict!", 
                                   font=("Arial", 16, "bold"), fg="#58A6FF", bg="#161B22")
        self.result_label.pack(pady=20)
        
        # History
        history_frame = tk.LabelFrame(left_frame, text="📜 Prediction History", 
                                     font=("Arial", 12, "bold"), fg="white", bg="#161B22")
        history_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=(0, 20))
        
        self.history_listbox = tk.Listbox(history_frame, font=("Consolas", 10), 
                                        bg="#0D1117", fg="#58A6FF", selectbackground="#00D4AA")
        history_scrollbar = ttk.Scrollbar(history_frame, orient=tk.VERTICAL, command=self.history_listbox.yview)
        self.history_listbox.configure(yscrollcommand=history_scrollbar.set)
        
        self.history_listbox.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        history_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        clear_btn = tk.Button(history_frame, text="🗑️ Clear", command=self.clear_history,
                             bg="#FF6B6B", fg="white", font=("Arial", 10))
        clear_btn.pack(pady=5)
        
        # Right panel - Dataset
        right_frame = tk.Frame(main_frame, bg="#161B22", relief=tk.RAISED, bd=2)
        right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)
        
        # Dataset frame
        dataset_frame = tk.LabelFrame(right_frame, text="📊 All Matches Dataset (200 records)", 
                                     font=("Arial", 14, "bold"), fg="white", bg="#161B22")
        dataset_frame.pack(fill=tk.BOTH, expand=True, padx=15, pady=20)
        
        # Treeview
        columns = ("Team1", "Team2", "Toss Winner", "Toss", "Venue", "Winner")
        self.tree = ttk.Treeview(dataset_frame, columns=columns, show="headings", height=22)
        
        for col in columns:
            self.tree.heading(col, text=col)
            self.tree.column(col, width=140, anchor=tk.CENTER)
        
        tree_scrollbar = ttk.Scrollbar(dataset_frame, orient=tk.VERTICAL, command=self.tree.yview)
        self.tree.configure(yscrollcommand=tree_scrollbar.set)
        
        self.tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        tree_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Load data button
        load_btn = tk.Button(dataset_frame, text="🔄 Refresh Dataset", command=self.load_data,
                           bg="#58A6FF", fg="white", font=("Arial", 11, "bold"))
        load_btn.pack(pady=10)
        
        # Stats
        stats_frame = tk.LabelFrame(right_frame, text="📈 Dataset Statistics", 
                                   font=("Arial", 12, "bold"), fg="white", bg="#161B22")
        stats_frame.pack(fill=tk.X, padx=15, pady=(0, 20))
        
        self.stats_label = tk.Label(stats_frame, text="", font=("Arial", 10), 
                                   fg="#58A6FF", bg="#161B22", justify=tk.LEFT)
        self.stats_label.pack(pady=10, padx=10)
        
        # Status
        self.status_var = tk.StringVar(value="🚀 Ready to predict IPL match winners!")
        status_label = tk.Label(right_frame, textvariable=self.status_var, 
                               relief=tk.SUNKEN, anchor=tk.W, fg="white", bg="#0D1117")
        status_label.pack(side=tk.BOTTOM, fill=tk.X, padx=15, pady=(0, 15))
        
        # Initialize
        self.load_data()
        self.update_stats()
    
    def load_data(self):
        for item in self.tree.get_children():
            self.tree.delete(item)
        
        original_df = pd.read_csv("matches.csv")
        for _, row in original_df.iterrows():
            self.tree.insert("", tk.END, values=(
                row['team1'], row['team2'], row['toss_winner'],
                row['toss_decision'], row['venue'], row['winner']
            ))
    
    def update_stats(self):
        df_stats = pd.read_csv("matches.csv")
        total_matches = len(df_stats)
        team_wins = df_stats['winner'].value_counts()
        top_team = team_wins.index[0]
        top_wins = team_wins.iloc[0]
        
        stats_text = f"""🏆 Total Matches: {total_matches}
 Most Wins: {top_team} ({top_wins} wins)
 Toss Impact: {len(df_stats[df_stats['toss_winner']==df_stats['winner']])/total_matches*100:.1f}% win toss & match
 Home Advantage: Analyze venue impact"""
        
        self.stats_label.config(text=stats_text)
    
    def predict_match(self):
        try:
            team1 = self.team1_var.get()
            team2 = self.team2_var.get()
            toss_winner = self.toss_var.get()
            toss_decision = self.toss_decision_var.get()
            venue = self.venue_var.get()
            
            if team1 == team2:
                raise ValueError("Team 1 and Team 2 cannot be the same!")
            
            winner = predict_match(team1, team2, toss_winner, toss_decision, venue)
            
            # Update result display
            win_color = "#00D4AA" if winner == team1 else "#FF6B6B"
            self.result_label.config(
                text=f"🏆 PREDICTED WINNER: {winner}",
                fg=win_color,
                font=("Arial", 18, "bold")
            )
            
            # Add to history
            timestamp = pd.Timestamp.now().strftime("%H:%M:%S")
            history_entry = f"{timestamp} | {team1} vs {team2} | {toss_winner} ({toss_decision}) @ {venue} → {winner}"
            self.history_listbox.insert(0, history_entry)
            if self.history_listbox.size() > 15:
                self.history_listbox.delete(15)
            
            # Status update
            self.status_var.set(f" Predicted: {winner} wins against {team1 if winner==team2 else team2}")
            
            # Celebration message
            messagebox.showinfo(" Prediction Result", 
                              f" IPL Match Prediction\n\n"
                              f"  {team1} vs {team2}\n"
                              f" Toss: {toss_winner} ({toss_decision})\n"
                              f"  Venue: {venue}\n\n"
                              f" **PREDICTED WINNER: {winner}**\n"
                              f" Model Confidence: High")
            
        except Exception as e:
            messagebox.showerror(" Prediction Error", f"Error: {str(e)}")
    
    def clear_history(self):
        self.history_listbox.delete(0, tk.END)

def main():
    root = tk.Tk()
    app = IPLPredictorApp(root)
    root.mainloop()

if __name__ == "__main__":
    main()

Dataset created successfully 
Training model...
Epoch 0, Loss: 2.0961
Epoch 10, Loss: 2.0782
Epoch 20, Loss: 2.0671
Epoch 30, Loss: 2.0612
Epoch 40, Loss: 2.0382

Model Accuracy: 22.50%
